# 06 数据并行 (DP to DDP)

## 简介

本笔记本介绍数据并行（Data Parallelism）的训练策略。
从朴素的模型复制（DP）到优化的 DistributedDataParallel（DDP），
我们将展示梯度同步、梯度累积和等效批次大小的概念。

## 1. 导入

In [ ]:
import torch
import torch.distributed as dist
from parallel.communication.setup import get_rank, get_world_size, init_process_group, cleanup
from parallel.communication.primitives import naive_all_reduce
from parallel.data_parallel.dp import sync_gradients_naive
from parallel.data_parallel.ddp import broadcast_model, gradient_bucket_sync
from parallel.data_parallel.gradient_accumulation import GradientAccumulator, compute_effective_batch_size

print(f"当前 rank: {get_rank()}")
print(f"world_size: {get_world_size()}")

### 🧠 直觉理解：数据并行

数据并行就像**多人同时做同一套试卷**：每个人（GPU）都有一份完整的参考书（模型副本），但每个人做不同的题目（不同数据）。做完后大家核对答案（All-Reduce 同步梯度），确保每个人的参考书保持一致。

**DP vs DDP 的区别**：DP 就像一个人收齐所有人的答案后统一批改（主从架构，master GPU 负载重）；DDP 就像每个人自己批改自己的，同时和别人交换批改结果（对称架构，通信与计算重叠）。

### 📐 数学原理：数据并行梯度同步

数据并行中，每个 GPU $i$ 在本地数据 $\mathcal{B}_i$ 上计算梯度：

$$g_i = \nabla_{\theta} \mathcal{L}(\theta; \mathcal{B}_i)$$

通过 All-Reduce 平均所有 GPU 的梯度：

$$\bar{g} = \frac{1}{P} \sum_{i=0}^{P-1} g_i$$

等价于在全局 batch $\mathcal{B} = \bigcup_i \mathcal{B}_i$ 上计算的梯度：

$$\bar{g} = \nabla_{\theta} \mathcal{L}(\theta; \mathcal{B})$$

**显存占用**：每 GPU 需存储完整模型 + 优化器状态 + 梯度 = $\Psi(1 + 2 + 1) = 4\Psi$（fp32 Adam），其中 $\Psi$ 为参数量。

## 2. 朴素数据并行 (DP) 的核心流程

```
                    GPU 0              GPU 1              GPU 2
                    ─────              ─────              ─────
                  batch_0             batch_1            batch_2
                     │                   │                  │
              forward(model)      forward(model)     forward(model)
                     │                   │                  │
             backward → grad_0  backward → grad_1  backward → grad_2
                     │                   │                  │
                     └───── All-Reduce (平均所有梯度) ─────┘
                     │                   │                  │
            optimizer.step()    optimizer.step()   optimizer.step()
                                (权重自动一致)
```

每张 GPU 持有**完整的模型副本**，处理**不同的数据**，通过 all-reduce 同步梯度。

缺点：每张卡都要存完整模型 + 优化器状态，无法训练超过单卡显存的模型。

In [ ]:
# 构建一个简单的 MLP 模型演示
model = torch.nn.Sequential(
    torch.nn.Linear(64, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10),
)

n_params = sum(p.numel() for p in model.parameters())
mem_fp32 = n_params * 4 / (1024 ** 2)  # MB
mem_fp16 = n_params * 2 / (1024 ** 2)
print(f"模型参数: {n_params:,}")
print(f"模型权重显存 (fp32): {mem_fp32:.2f} MB")
print(f"模型权重显存 (fp16): {mem_fp16:.2f} MB")
print(f"优化器显存 (Adam, fp32): ~{mem_fp32*2:.2f} MB (moment + variance)")
print(f"\n总显存 (不含激活): {mem_fp32*(1+2):.2f} MB / GPU")
print(f"注意: 每张 GPU 都需要相同的显存量!")

## 3. Gradient Bucketing (分桶同步)

DDP 的一个关键优化：将多个小参数的梯度打包成一个 bucket，
批量通信以减少 all-reduce 的启动延迟开销。

默认 bucket_size=25MB 是 PyTorch DDP 的典型设置。

In [ ]:
# 统计梯度张量大小
print("模型各参数的梯度大小:")
for name, param in model.named_parameters():
    size_kb = param.numel() * 4 / 1024
    print(f"  {name:20s}: {list(param.shape)}  ->  {size_kb:.2f} KB")

total_grad_kb = sum(p.numel() * 4 / 1024 for p in model.parameters())
print(f"\n总梯度大小: {total_grad_kb:.2f} KB")
print(f"\n如果 bucket_size = 25MB ({25*1024:.0f} KB), 所有梯度可以合并为 1 个 bucket")
print(f"这样 1 次 all-reduce 就够了, 而不是 {sum(1 for _ in model.parameters())} 次")

## 4. DDP 相比 DP 的改进总结

| 改进项 | 朴素 DP | DDP |
|---|---|---|
| **模型同步** | 每次 forward 前 broadcast 模型 | 只在初始化时 broadcast 一次 |
| **梯度同步** | 逐参数 all-reduce | 分桶批量 all-reduce |
| **通信时机** | forward 后才开始通信 | Backward 中重叠通信和计算 |
| **实现** | 主从架构，master GPU 负载高 | 对所有 GPU 对称，无瓶颈 |

### 📊 DP/DDP/FSDP 显存对比柱状图

对比不同数据并行策略下每张 GPU 的显存占用。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 假设 7B 参数模型，fp16 参数量 = 14GB
params_gb = 14  # fp16 参数
optimizer_gb = params_gb * 2  # Adam: moment + variance (fp32)
gradients_gb = params_gb  # 梯度 (fp16)

# DP/DDP: 每卡存完整模型
dp_total = params_gb + optimizer_gb + gradients_gb

# FSDP ZeRO-1: 分片优化器状态
P = 8
zero1_total = params_gb + optimizer_gb / P + gradients_gb

# FSDP ZeRO-2: 分片优化器 + 梯度
zero2_total = params_gb + optimizer_gb / P + gradients_gb / P

# FSDP ZeRO-3: 分片所有
zero3_total = params_gb / P + optimizer_gb / P + gradients_gb / P

labels = ['DP/DDP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3']
totals = [dp_total, zero1_total, zero2_total, zero3_total]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

fig, ax = plt.subplots(figsize=(10, 6))

# 堆叠柱状图
params_vals = [params_gb, params_gb, params_gb, params_gb / P]
opt_vals = [optimizer_gb, optimizer_gb / P, optimizer_gb / P, optimizer_gb / P]
grad_vals = [gradients_gb, gradients_gb, gradients_gb / P, gradients_gb / P]

x = np.arange(len(labels))
width = 0.5

ax.bar(x, params_vals, width, label='参数', color='#3498db', edgecolor='black', linewidth=0.5)
ax.bar(x, opt_vals, width, bottom=params_vals, label='优化器状态', color='#e74c3c', edgecolor='black', linewidth=0.5)
bottoms = [p + o for p, o in zip(params_vals, opt_vals)]
ax.bar(x, grad_vals, width, bottom=bottoms, label='梯度', color='#2ecc71', edgecolor='black', linewidth=0.5)

for i, total in enumerate(totals):
    ax.text(i, total + 1, f'{total:.1f} GB', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('每 GPU 显存 (GB)', fontsize=12)
ax.set_title(f'数据并行显存对比 (7B 模型, P={P} GPUs)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 ZeRO 分级示意图

ZeRO（Zero Redundancy Optimizer）通过分片不同级别的状态来减少冗余。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

P = 4  # 4 个 GPU
states = ['参数', '梯度', '优化器']
state_colors = {'参数': '#3498db', '梯度': '#2ecc71', '优化器': '#e74c3c'}

zero_configs = [
    ('DP/DDP\n(无分片)', [True, True, True]),
    ('ZeRO-1\n(分片优化器)', [True, True, False]),
    ('ZeRO-2\n(分片优化器+梯度)', [True, False, False]),
    ('ZeRO-3\n(分片全部)', [False, False, False]),
]

for ax, (title, replicated) in zip(axes, zero_configs):
    for gpu in range(P):
        y = gpu * 3
        for s, state in enumerate(states):
            color = state_colors[state] if replicated[s] else '#bdc3c7'
            alpha = 0.8 if replicated[s] else 0.3
            rect = mpatches.FancyBboxPatch((0.5, y + s * 0.9), 3, 0.8,
                                            boxstyle='round,pad=0.05',
                                            facecolor=color, alpha=alpha, edgecolor='gray')
            ax.add_patch(rect)
            label = state if replicated[s] else f'{state}(分片)'
            ax.text(2, y + s * 0.9 + 0.4, label, ha='center', va='center', fontsize=7)
        ax.text(-0.3, y + 1.2, f'GPU {gpu}', ha='right', va='center', fontsize=8, fontweight='bold')

    ax.set_xlim(-1.5, 4.5)
    ax.set_ylim(-0.5, P * 3)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('ZeRO 分级分片策略示意图', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. 梯度累积和等效 Batch Size

梯度累积让我们在小显存 GPU 上训练大 batch：
模拟 batch_size=256 但每步只跑 64，累积 4 步后同步一次。

In [ ]:
model = torch.nn.Linear(4, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

accumulation_steps = 4
micro_batch_size = 8
world_size = 2  # 假设 2 卡 DP

acc = GradientAccumulator(model, accumulation_steps)

print("梯度累积演示:")
print(f"  micro_batch_size = {micro_batch_size}")
print(f"  accumulation_steps = {accumulation_steps}")
print(f"  world_size (DP) = {world_size}")

effective = compute_effective_batch_size(micro_batch_size, accumulation_steps, world_size)
print(f"\n  等效全局 batch size = {micro_batch_size} × {accumulation_steps} × {world_size} = {effective}")
print(f"\n  即: 每张卡的"物理"batch size = {micro_batch_size}")
print(f"      累积 {accumulation_steps} 步才更新一次参数")
print(f"      {world_size} 张卡一起跑 → 等效于单卡 batch_size={effective}")

# 模拟训练
print(f"\n模拟 {accumulation_steps} 步训练:")
for i in range(accumulation_steps):
    x = torch.randn(micro_batch_size, 4)
    target = torch.randn(micro_batch_size, 2)
    loss = torch.nn.functional.mse_loss(model(x), target)
    loss.backward()  # 梯度累积 (PyTorch 默认累加到 .grad)

    should_update = acc.step()
    print(f"  Step {i+1}: loss={loss.item():.4f}, 需要更新={should_update}")
    if should_update:
        optimizer.step()
        optimizer.zero_grad()
        print(f"         → 梯度已除以 {accumulation_steps}, 执行 optimizer.step()")

## 6. 通信开销分析

在 DP 中，每个 iteration 需要进行一次 all-reduce 来同步梯度。
通信时间取决于梯度大小和带宽。

In [ ]:
def dp_communication_overhead(model_size_gb, world_size, bandwidth_gbps=100):
    """
    计算 DP 梯度同步的通信开销。
    Ring All-Reduce 总通信量 ≈ 2 × model_size (每个参数 1 次 send + 1 次 recv)
    """
    total_data_tx_gb = 2 * model_size_gb  # Ring 总通信量
    time_per_device_ms = total_data_tx_gb / (bandwidth_gbps / 8) * 1000
    return time_per_device_ms

# 常见模型规模
models = [
    ("Llama-7B", 7 * 2 / 1024),    # 7B fp16 = 14GB
    ("Llama-13B", 13 * 2 / 1024),
    ("Llama-70B", 70 * 2 / 1024),
    ("DeepSeek-V3", 671 * 2 / 1024),
]

print("DP 梯度同步通信时间 (Ring All-Reduce, 100Gbps 带宽):")
print(f"{'模型':<20s} {'大小(GB)':>10s} {'通信(ms)':>10s}")
print("-" * 40)
for name, size_gb in models:
    t = dp_communication_overhead(size_gb, world_size=8)
    print(f"{name:<20s} {size_gb:>10.2f} {t:>10.1f}")

print("\n★ 大模型 (70B+) 的梯度同步需要数百毫秒")
print("★ 梯度累积可以减少通信频率 (每 N 步才同步一次)")
print("★ 但梯度累积不能无限增大 (可能导致训练不稳定)")

## 7. 关键要点总结

1. **朴素 DP**: 每 GPU 完整模型副本，backward 后 all-reduce 梯度，简单但显存效率低
2. **DDP**: 在 DP 基础上加入梯度分桶、通信计算重叠，大大减少通信开销
3. **Gradient Bucketing**: 将多个小梯度打包成一个 bucket 一起通信，减少 all-reduce 启动延迟
4. **梯度累积**: 多次 micro-batch 后才做一次 optimizer.step()，等效于大 batch 训练
5. **等效 Batch Size** = micro_batch_size × accumulation_steps × world_size
6. **通信开销**: DP 无法处理超过单卡显存的模型，需要结合 TP/PP 等策略

## 📝 练习题

### 思考题
1. 为什么 DDP 的 Gradient Bucketing 能减少通信开销？如果所有参数的梯度大小相同，Bucketing 还有用吗？

### 编程题
2. 实现一个简单的梯度累积训练循环：给定 micro_batch_size=4, accumulation_steps=8, world_size=2，计算等效 batch_size 并验证梯度累积后的参数更新与一次性大 batch 更新等价。

### 分析题
3. 一个 70B 参数模型（fp16 约 140GB），在 8×80GB GPU 上使用 ZeRO-3 训练。计算每 GPU 的显存占用（参数+优化器+梯度），并分析剩余显存能否支持 seq_len=4096, batch_size=1 的训练。